In [11]:
import pandas as pd
import numpy as np
import json
import os
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from openai import OpenAI


In [12]:
# Load API key
with open("key.txt", "r") as f:
    api_key = f.read().strip()

client = OpenAI(api_key=api_key)


In [13]:
import re

# Arabic tashkeel unicode range
TASHKEEL_REGEX = re.compile(r'[\u0617-\u061A\u064B-\u0652]')

def remove_tashkeel(text):
    if not isinstance(text, str):
        return text
    return re.sub(TASHKEEL_REGEX, '', text)


In [14]:
df_pos = pd.read_csv("tashbeeh_dataset.csv")
df_neg = pd.read_csv("non_tashbeeh_random_sample.csv")

df_pos["label"] = 1
df_neg["label"] = 0

TEXT_COL = "البيت_الكامل"

df = pd.concat([df_pos, df_neg], ignore_index=True)
df = df[[TEXT_COL, "label"]].dropna()

# 🔥 REMOVE TASHKEEL
df[TEXT_COL] = df[TEXT_COL].apply(remove_tashkeel)

df.head()


,البيت_الكامل,label
0,أهم بشيء والليالي كأنها تطاردني عن كونه وأطارد,1
1,تثنى على قدر الطعان كأنما مفاصلها تحت الرماح م...,1
2,ولما رأيت الناس دون محله تيقنت أن الدهر للناس ...,1
3,مخضبة والقوم صرعى كأنها وإن لم يكونوا ساجدين م...,1
4,تنكسهم والسابقات جبالهم وتطعن فيهم والرماح الم...,1


In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    df[TEXT_COL].values,
    df["label"].values,
    test_size=0.05,
    stratify=df["label"],
    random_state=42
)

len(X_train), len(X_test)


(1035, 55)

In [16]:
SYSTEM_PROMPT = """
You are an expert in Arabic rhetoric (البلاغة العربية).
Your task is to determine whether a verse contains TASHBEEH (تشبيه).

Rules:
- Output ONLY one number:
    1 → contains tashbeeh
    0 → does NOT contain tashbeeh
- No explanations.
- No extra text.
"""


In [17]:
def build_prompt(verse):
    return f"""
Verse:
{verse}

Question:
Does this verse contain tashbeeh?
Answer (0 or 1):
"""


In [18]:
def predict_tashbeeh(verse):
    response = client.responses.create(
        model="gpt-5.2-pro-2025-12-11",
        input=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": build_prompt(verse)
            }
        ]
    )

    # Extract text safely
    out = response.output_text.strip()
    return 1 if out.startswith("1") else 0


In [19]:
# def predict_tashbeeh(verse):
#     response = client.chat.completions.create(
#         model="gpt-5.2-2025-12-11",
#         messages=[
#             {"role": "system", "content": SYSTEM_PROMPT},
#             {"role": "user", "content": build_prompt(verse)}
#         ]
#         # ❌ temperature REMOVED
#     )

#     out = response.choices[0].message.content.strip()
#     return 1 if out == "1" else 0


In [20]:
y_pred = []

for verse in tqdm(X_test):
    try:
        y_pred.append(predict_tashbeeh(verse))
    except Exception as e:
        print("Error:", e)
        y_pred.append(0)  # safe fallback


100%|██████████| 55/55 [12:59<00:00, 14.17s/it]


In [21]:
acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("Accuracy:", acc)
print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Non-Tashbeeh", "Tashbeeh"]))


Accuracy: 0.8727272727272727

Confusion Matrix:
[[23  4]
 [ 3 25]]

Classification Report:
              precision    recall  f1-score   support

Non-Tashbeeh       0.88      0.85      0.87        27
    Tashbeeh       0.86      0.89      0.88        28

    accuracy                           0.87        55
   macro avg       0.87      0.87      0.87        55
weighted avg       0.87      0.87      0.87        55



In [22]:
# ===============================
# Save GPT predictions to CSV
# ===============================

results_df = pd.DataFrame({
    "verse": X_test,
    "y_true": y_test,
    "y_pred": y_pred
})

OUT_PATH = "gpt_tashbeeh_predictions_test.csv"
results_df.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

print(f"Saved results to: {OUT_PATH}")
results_df.head()


Saved results to: gpt_tashbeeh_predictions_test.csv


,verse,y_true,y_pred
0,فكر محمية من ان يفر ، كما كر المحامي حفاظا ، ...,1,1
1,ليزد بنو الحسن الشراف تواضعا هيهات تكتم في الظ...,1,1
2,له أيطلا ظبي وساقا نعامة وإرخاء سرحان وتقريب تتفل,1,1
3,لها فخذان أكمل النخض فيهما كأنهما بابا منيف ممرد,1,1
4,وتكسر سرجها أبدا شماسا وتسقط في الوحول وفي الرمال,0,0
